## Setting up variables

In [ ]:
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 25.7 MB/s eta 0:00:00


In [ ]:
import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig
from datasets import load_dataset, Dataset, DatasetDict
import wandb
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datetime import datetime
import matplotlib.pyplot as plt

In [4]:
# Constants

BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"
PROJECT_NAME = "gear_training"
HF_USER = "yrichard"
DATA_USER = "yrichard"
DATASET_NAME = f"{DATA_USER}/gear_raw"

MAX_SEQUENCE_LENGTH = 768
RUN_NAME = "2026-04-28_13.15.01"

PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

### Log in to HuggingFace & Load dataset

In [ ]:
# Log in to HuggingFace

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
dataset = load_dataset(DATASET_NAME)
train = dataset['train']
val = dataset['validation']
test = dataset['test']

README.md:   0%|          | 0.00/727 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/70.8k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/17.6k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/14.8k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/50 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/28 [00:00<?, ? examples/s]

## Testing unmerged model

In [ ]:
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )

In [ ]:
# Load the Tokenizer and the Model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Memory footprint: 2197.6 MB


In [ ]:
from peft import PeftModel
import torch

fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)

# Explicitly cast the lm_head to float32 to resolve type mismatch.
# The error "expected scalar type Float but found BFloat16" indicates lm_head expects float32
# but the input it receives (or its own weights) are in bfloat16. Casting to float32 ensures
# consistency for this critical layer.
if hasattr(fine_tuned_model.base_model, 'lm_head') and fine_tuned_model.base_model.lm_head.weight.dtype != torch.float32:
    print(f"Casting lm_head from {fine_tuned_model.base_model.lm_head.weight.dtype} to torch.float32...")
    fine_tuned_model.base_model.lm_head = fine_tuned_model.base_model.lm_head.to(torch.float32)

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/48.7M [00:00<?, ?B/s]

Casting lm_head from torch.bfloat16 to torch.float32...


## Testing merged model

In [ ]:
MERGED_PROJECT_RUN_NAME = f"{PROJECT_RUN_NAME}-merged"
HUB_MODEL_NAME = f"{HF_USER}/{MERGED_PROJECT_RUN_NAME}"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

fine_tuned_model = AutoModelForCausalLM.from_pretrained(
    HUB_MODEL_NAME,
    device_map="auto",
)
fine_tuned_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Memory footprint: 6425.5 MB


## Infer model

In [ ]:
def model_predict(item):
    inputs = tokenizer(item["prompt"],return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=MAX_SEQUENCE_LENGTH)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)

In [ ]:
gear = test[7]
print(gear["prompt"])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 28 Apr 2026

You are a mountain climbing equipment assistant. Parse the following gear description and return a JSON array. Each element must have exactly three fields:
	- "name": equipment name (string, in french)
	- "quantity": number needed (integer, 1 if unspecified)
	- "notes": "optional" or "mandatory" (translated in french), plus any relevant detail (string, in french)
The name of these equipments are related with the mountain activities. You should only point out personal equipment, for instance quickdraws or rope.
You should include only equipment you're absolutely sure about. Output ONLY the JSON array, no explanation.<|eot_id|><|start_header_id|>user<|end_header_id|>

Gear description:
 2 jeux complets Friends jusqu’au 4.
1 numéro 5 (facultatif).
1 jeu de câblés.
Corde 50 m 
1 Marteau et 4-5 pitons (au cas où)
Gants de fissure (facultatif)<|eot_id|><|start_header_id

In [ ]:
print("Answer:")
model_predict(gear)

Answer:


KeyboardInterrupt: 

## Evaluate model quality

In [ ]:
import json
import re
import math
from difflib import SequenceMatcher
from itertools import accumulate

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from tqdm.auto import tqdm
from IPython.display import clear_output


GREEN = "\033[92m"
YELLOW = "\033[93m"
RED = "\033[91m"
RESET = "\033[0m"
COLOR_MAP = {"red": RED, "orange": YELLOW, "green": GREEN}

DEFAULT_SIZE = 200


def parse_json(value: str) -> list[dict] | None:
    """Extract and parse a JSON array from raw model output."""
    if isinstance(value, list):
        return value
    try:
        return json.loads(value)
    except Exception:
        pass
    match = re.search(r"\[.*\]", value, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except Exception:
            pass
    return None


def _name_similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, a.lower().strip(), b.lower().strip()).ratio()


def _match_items(pred: list[dict], truth: list[dict], threshold: float = 0.6) -> tuple[int, int, int]:
    """Greedy name-based matching. Returns (tp, fp, fn)."""
    matched = set()
    tp = 0
    for p in pred:
        p_name = p.get("name", "")
        best, best_j = 0.0, -1
        for j, t in enumerate(truth):
            if j in matched:
                continue
            score = _name_similarity(p_name, t.get("name", ""))
            if score > best:
                best, best_j = score, j
        if best >= threshold:
            tp += 1
            matched.add(best_j)
    return tp, len(pred) - tp, len(truth) - tp


def _f1(tp: int, fp: int, fn: int) -> tuple[float, float, float]:
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return f1, precision, recall


class JsonTester:
    def __init__(self, predictor, data, title=None, size=DEFAULT_SIZE):
        self.predictor = predictor
        self.data = data
        self.title = title or self.make_title(predictor)
        self.size = min(size, len(data))
        self.titles: list[str] = []
        self.f1s: list[float] = []
        self.precisions: list[float] = []
        self.recalls: list[float] = []
        self.valid_json: list[bool] = []
        self.colors: list[str] = []

    @staticmethod
    def make_title(predictor) -> str:
        return predictor.__name__.replace("__", ".").replace("_", " ").title().replace("Gpt", "GPT")

    def color_for(self, f1: float) -> str:
        if f1 >= 0.8:
            return "green"
        elif f1 >= 0.5:
            return "orange"
        else:
            return "red"

    def run_datapoint(self, i: int):
        datapoint = self.data[i]
        raw = self.predictor(datapoint)

        pred = parse_json(raw)
        is_valid = pred is not None
        pred = pred or []

        truth_raw = datapoint["completion"]
        truth = json.loads(truth_raw) if isinstance(truth_raw, str) else truth_raw

        tp, fp, fn = _match_items(pred, truth)
        f1, precision, recall = _f1(tp, fp, fn)
        color = self.color_for(f1)

        pieces = datapoint["prompt"].split("Gear description:")
        snippet = pieces[1].strip() if len(pieces) > 1 else datapoint["prompt"]
        title = snippet[:40] + "..." if len(snippet) > 40 else snippet

        return title, f1, precision, recall, is_valid, color

    def chart(self, title: str):
        df = pd.DataFrame({
            "f1": self.f1s,
            "precision": self.precisions,
            "recall": self.recalls,
            "title": self.titles,
            "color": self.colors,
        })
        df["hover"] = [
            f"{t}<br>F1={f:.2f}  P={p:.2f}  R={r:.2f}"
            for t, f, p, r in zip(df["title"], df["f1"], df["precision"], df["recall"])
        ]

        fig = px.histogram(
            df,
            x="f1",
            color="color",
            color_discrete_map={"green": "green", "orange": "orange", "red": "red"},
            title=title,
            labels={"f1": "F1 Score"},
            nbins=20,
            width=800,
            height=400,
        )
        fig.update_layout(bargap=0.1, showlegend=False, xaxis_range=[0, 1])
        fig.show()

    def error_trend_chart(self):
        errors = [1.0 - f for f in self.f1s]
        n = len(errors)
        x = list(range(1, n + 1))

        running_sums = list(accumulate(errors))
        running_means = [s / i for s, i in zip(running_sums, x)]

        running_squares = list(accumulate(e * e for e in errors))
        running_stds = [
            math.sqrt((sq / i) - (m ** 2)) if i > 1 else 0
            for i, sq, m in zip(x, running_squares, running_means)
        ]
        ci = [1.96 * (sd / math.sqrt(i)) if i > 1 else 0 for i, sd in zip(x, running_stds)]

        final_mean, final_ci = running_means[-1], ci[-1]
        title = f"{self.title} – Avg Error (1−F1): {final_mean:.3f} ± {final_ci:.3f}"

        upper = [m + c for m, c in zip(running_means, ci)]
        lower = [m - c for m, c in zip(running_means, ci)]

        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=x + x[::-1],
            y=upper + lower[::-1],
            fill="toself",
            fillcolor="rgba(128,128,128,0.2)",
            line=dict(color="rgba(255,255,255,0)"),
            hoverinfo="skip",
            showlegend=False,
        ))
        fig.add_trace(go.Scatter(
            x=x,
            y=running_means,
            mode="lines",
            line=dict(width=3, color="firebrick"),
            name="Cumulative Avg Error",
            customdata=list(zip(ci)),
            hovertemplate="n=%{x}<br>Avg 1−F1=%{y:.3f}<br>±95% CI=%{customdata[0]:.3f}<extra></extra>",
        ))
        fig.update_layout(
            title=title,
            xaxis_title="Number of Datapoints",
            yaxis_title="Error (1 − F1)",
            width=800,
            height=300,
            template="plotly_white",
            showlegend=False,
        )
        fig.show()

    def report(self):
        avg_f1 = sum(self.f1s) / len(self.f1s)
        avg_p = sum(self.precisions) / len(self.precisions)
        avg_r = sum(self.recalls) / len(self.recalls)
        json_rate = sum(self.valid_json) / len(self.valid_json) * 100
        title = (
            f"{self.title} results<br>"
            f"<b>F1:</b> {avg_f1:.3f}  "
            f"<b>Precision:</b> {avg_p:.3f}  "
            f"<b>Recall:</b> {avg_r:.3f}  "
            f"<b>Valid JSON:</b> {json_rate:.1f}%"
        )
        self.error_trend_chart()
        self.chart(title)

    def run(self):
        for i in tqdm(range(self.size)):
            title, f1, precision, recall, is_valid, color = self.run_datapoint(i)
            self.titles.append(title)
            self.f1s.append(f1)
            self.precisions.append(precision)
            self.recalls.append(recall)
            self.valid_json.append(is_valid)
            self.colors.append(color)
            print(f"{COLOR_MAP[color]}F1={f1:.2f} ", end="")
        clear_output(wait=True)
        self.report()


def evaluate(function, data, size=DEFAULT_SIZE):
    JsonTester(function, data, size=size).run()


In [ ]:
set_seed(42)
evaluate(model_predict, test)